# CO2 Awareness Experiment - Data Analysis

This notebook analyzes data from participants in the CO2 awareness experiment.

**Key Research Questions:**
1. Did participants improve their CO2 knowledge after the learning intervention?
2. Does the effect differ by demographics (age, gender, diet)?
3. What do participants report learning from the experience?

**Data Structure:**
- 6 quiz scores (pre/post for generic, AH-specific, and personal products)
- Demographics (age, gender, diet, shopping frequency)
- Likert questionnaires (pre & post attitudes)
- Open-ended reflection responses

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import requests
import json
from datetime import datetime

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

print("Libraries loaded successfully!")

## 2. Load Data from API

Fetch experiment data from the local server. Make sure the server is running (`npm run dev` in sustainable-shop-webapp).

In [ ]:
# Configuration
SERVER_URL = "http://localhost:3001"
COMPLETED_ONLY = True  # Set to False to include incomplete sessions

# Fetch data from API
def fetch_data(completed_only=True):
    url = f"{SERVER_URL}/api/admin/experiment/export"
    params = {'completedOnly': 'true'} if completed_only else {}
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.ConnectionError:
        print(f"❌ Could not connect to server at {SERVER_URL}")
        print("   Make sure the server is running: cd sustainable-shop-webapp && npm run dev")
        return None

# Load data
raw_data = fetch_data(COMPLETED_ONLY)

if raw_data:
    print(f"✅ Loaded {raw_data['count']} sessions ({raw_data.get('completedCount', 'N/A')} completed)")
    print(f"   Exported at: {raw_data['exportedAt']}")

In [ ]:
# Convert to DataFrame
sessions = raw_data.get('sessions', [])

# Extract flat fields (exclude raw JSONB columns starting with _)
flat_data = []
for s in sessions:
    row = {k: v for k, v in s.items() if not k.startswith('_')}
    flat_data.append(row)

df = pd.DataFrame(flat_data)

# Convert timestamps
for col in ['started_at', 'completed_at', 'updated_at']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"DataFrame shape: {df.shape}")
df.head()

## 3. Data Inspection

Understand the structure and quality of the data.

In [ ]:
# Basic info about the dataset
print("=== Dataset Info ===")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumn types:")
df.info()

In [ ]:
# Check key columns for missing values
key_columns = ['quiz1_score', 'quiz2_score', 'quiz3_score', 'quiz4_score', 
               'quiz5_score', 'quiz6_score', 'demo_age', 'demo_gender', 'demo_diet']

print("=== Missing Values in Key Columns ===")
missing = df[key_columns].isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values in key columns!")

print("\n=== Quiz Score Ranges ===")
quiz_cols = [c for c in df.columns if 'quiz' in c and 'score' in c]
df[quiz_cols].describe().round(1)

## 4. Demographics Overview

Understand who participated in the experiment.

In [ ]:
# Demographics breakdown
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Gender distribution
if 'demo_gender' in df.columns:
    gender_counts = df['demo_gender'].value_counts()
    axes[0, 0].pie(gender_counts, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
    axes[0, 0].set_title('Gender Distribution')

# Diet distribution
if 'demo_diet' in df.columns:
    diet_counts = df['demo_diet'].value_counts()
    sns.barplot(x=diet_counts.index, y=diet_counts.values, ax=axes[0, 1], palette='viridis')
    axes[0, 1].set_title('Diet Distribution')
    axes[0, 1].set_xlabel('Diet Type')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].tick_params(axis='x', rotation=45)

# Age distribution
if 'demo_age' in df.columns:
    df['demo_age_numeric'] = pd.to_numeric(df['demo_age'], errors='coerce')
    ages = df['demo_age_numeric'].dropna()
    axes[1, 0].hist(ages, bins=15, edgecolor='black', alpha=0.7)
    axes[1, 0].set_title(f'Age Distribution (mean={ages.mean():.1f}, n={len(ages)})')
    axes[1, 0].set_xlabel('Age')
    axes[1, 0].set_ylabel('Count')

# Shopping frequency
if 'demo_shopping_frequency' in df.columns:
    freq_counts = df['demo_shopping_frequency'].value_counts()
    sns.barplot(x=freq_counts.index, y=freq_counts.values, ax=axes[1, 1], palette='coolwarm')
    axes[1, 1].set_title('Shopping Frequency')
    axes[1, 1].set_xlabel('Frequency')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Summary table
print("\n=== Demographics Summary ===")
for col in ['demo_gender', 'demo_diet', 'demo_shopping_frequency']:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts())

## 5. Learning Effect Analysis (Pre vs Post)

**Key hypothesis:** Participants score higher on post-intervention quizzes than pre-intervention quizzes.

Quiz pairs:
- **Generic products:** Quiz 1 (pre) → Quiz 3 (post)
- **AH-specific products:** Quiz 5 (pre) → Quiz 6 (post)  
- **Personal products:** Quiz 2 (pre) → Quiz 4 (post)

We use **paired t-tests** since each participant has both pre and post scores.

In [ ]:
def analyze_learning_effect(df, pre_col, post_col, label):
    """Perform paired t-test and calculate effect size for pre/post comparison."""
    # Get paired data (both pre and post available)
    paired = df[[pre_col, post_col]].dropna()
    
    if len(paired) < 3:
        return {'label': label, 'error': f'Insufficient data (n={len(paired)})'}
    
    pre = paired[pre_col]
    post = paired[post_col]
    
    # Paired t-test
    t_stat, p_value = stats.ttest_rel(post, pre)
    
    # Wilcoxon signed-rank test (non-parametric)
    try:
        w_stat, w_pvalue = stats.wilcoxon(post - pre)
    except:
        w_stat, w_pvalue = None, None
    
    # Effect size (Cohen's d for paired samples)
    diff = post - pre
    cohens_d = diff.mean() / diff.std() if diff.std() > 0 else 0
    
    return {
        'label': label,
        'n': len(paired),
        'pre_mean': pre.mean(),
        'pre_std': pre.std(),
        'post_mean': post.mean(),
        'post_std': post.std(),
        'improvement': post.mean() - pre.mean(),
        't_statistic': t_stat,
        'p_value': p_value,
        'wilcoxon_p': w_pvalue,
        'cohens_d': cohens_d,
        'significant': p_value < 0.05
    }

# Analyze all three quiz pairs
quiz_pairs = [
    ('quiz1_score', 'quiz3_score', 'Generic Products'),
    ('quiz5_score', 'quiz6_score', 'AH-Specific Products'),
    ('quiz2_score', 'quiz4_score', 'Personal Products'),
]

results = []
for pre, post, label in quiz_pairs:
    result = analyze_learning_effect(df, pre, post, label)
    results.append(result)

# Display results
results_df = pd.DataFrame(results)
print("=" * 80)
print("LEARNING EFFECT ANALYSIS - PAIRED T-TEST RESULTS")
print("=" * 80)

for r in results:
    if 'error' in r:
        print(f"\n{r['label']}: {r['error']}")
        continue
    
    sig_marker = "***" if r['p_value'] < 0.001 else "**" if r['p_value'] < 0.01 else "*" if r['p_value'] < 0.05 else "ns"
    effect_size = "large" if abs(r['cohens_d']) >= 0.8 else "medium" if abs(r['cohens_d']) >= 0.5 else "small" if abs(r['cohens_d']) >= 0.2 else "negligible"
    
    print(f"\n{'─' * 60}")
    print(f"📊 {r['label']} (n={r['n']})")
    print(f"{'─' * 60}")
    print(f"   Pre-test:     {r['pre_mean']:.1f} ± {r['pre_std']:.1f}")
    print(f"   Post-test:    {r['post_mean']:.1f} ± {r['post_std']:.1f}")
    print(f"   Improvement:  {r['improvement']:+.1f} points")
    print(f"")
    print(f"   t-statistic:  {r['t_statistic']:.3f}")
    print(f"   p-value:      {r['p_value']:.4f} {sig_marker}")
    print(f"   Cohen's d:    {r['cohens_d']:.3f} ({effect_size} effect)")

print(f"\n{'=' * 80}")
print("Significance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

## 6. Data Visualization

Visual comparison of pre vs post quiz scores.

In [ ]:
# Box plots comparing pre vs post scores
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

quiz_pairs = [
    ('quiz1_score', 'quiz3_score', 'Generic Products'),
    ('quiz5_score', 'quiz6_score', 'AH-Specific Products'),
    ('quiz2_score', 'quiz4_score', 'Personal Products'),
]

colors = ['#ff6b6b', '#4ecdc4']  # red for pre, teal for post

for ax, (pre, post, title) in zip(axes, quiz_pairs):
    data_to_plot = []
    labels = []
    
    pre_data = df[pre].dropna()
    post_data = df[post].dropna()
    
    bp = ax.boxplot([pre_data, post_data], labels=['Pre', 'Post'], patch_artist=True)
    
    # Color the boxes
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    # Add mean markers
    ax.scatter([1, 2], [pre_data.mean(), post_data.mean()], 
               color='black', marker='D', s=50, zorder=3, label='Mean')
    
    # Calculate improvement
    improvement = post_data.mean() - pre_data.mean()
    ax.set_title(f'{title}\n(+{improvement:.1f} points)', fontsize=12)
    ax.set_ylabel('Score (0-100)')
    ax.set_ylim(0, 105)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.3, label='Random (50)')

plt.suptitle('Pre vs Post Intervention Quiz Scores', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Individual improvement distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

improvement_cols = [
    ('generic_improvement', 'Generic Products'),
    ('ah_improvement', 'AH-Specific Products'),
    ('personal_improvement', 'Personal Products'),
]

for ax, (col, title) in zip(axes, improvement_cols):
    if col in df.columns:
        data = df[col].dropna()
        
        # Histogram
        ax.hist(data, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
        ax.axvline(x=0, color='red', linestyle='--', linewidth=2, label='No change')
        ax.axvline(x=data.mean(), color='green', linestyle='-', linewidth=2, 
                   label=f'Mean: {data.mean():.1f}')
        
        ax.set_title(f'{title}\n(n={len(data)})')
        ax.set_xlabel('Score Improvement (Post - Pre)')
        ax.set_ylabel('Count')
        ax.legend()

plt.suptitle('Distribution of Individual Learning Effects', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# What percentage improved?
print("\n=== Percentage of Participants Who Improved ===")
for col, label in improvement_cols:
    if col in df.columns:
        data = df[col].dropna()
        pct_improved = (data > 0).mean() * 100
        pct_same = (data == 0).mean() * 100  
        pct_worse = (data < 0).mean() * 100
        print(f"{label}: {pct_improved:.1f}% improved, {pct_same:.1f}% same, {pct_worse:.1f}% worse")

## 7. Learning Effect by Demographics

Does the intervention work better for certain groups?

In [ ]:
# Learning effect by diet type
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

improvement_cols = ['generic_improvement', 'ah_improvement', 'personal_improvement']
titles = ['Generic Products', 'AH-Specific Products', 'Personal Products']

for ax, col, title in zip(axes, improvement_cols, titles):
    if col in df.columns and 'demo_diet' in df.columns:
        # Group by diet
        diet_data = df.groupby('demo_diet')[col].agg(['mean', 'std', 'count']).reset_index()
        diet_data = diet_data[diet_data['count'] >= 2]  # At least 2 participants
        
        if len(diet_data) > 0:
            bars = ax.bar(diet_data['demo_diet'], diet_data['mean'], 
                         yerr=diet_data['std'], capsize=5, 
                         color='steelblue', alpha=0.7, edgecolor='black')
            
            # Add count labels
            for i, (idx, row) in enumerate(diet_data.iterrows()):
                ax.text(i, row['mean'] + row['std'] + 2, f"n={int(row['count'])}", 
                       ha='center', fontsize=9)
            
            ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
            ax.set_title(title)
            ax.set_xlabel('Diet Type')
            ax.set_ylabel('Mean Improvement')
            ax.tick_params(axis='x', rotation=45)

plt.suptitle('Learning Effect by Diet Type', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Questionnaire Analysis

Analyze attitudes and self-perceptions before and after the intervention.

**Pre-questionnaire (1-5 Likert):**
- pre_q1: I consider my food choices environmentally sustainable
- pre_q2: I feel confident in my knowledge of food environmental impact
- pre_q3: I trust eco-labels when buying food
- pre_q4: Eco-labels are easy to understand
- pre_q5: I find it easy to compare products by environmental impact
- pre_q6: I actively consider environmental impact when buying food

**Post-questionnaire (1-5 Likert):**
- post_q1: Better understanding of environmental impact after study
- post_q2: The ranking system was clear and easy to understand
- post_q3: I trust the CO₂ ranking system presented
- post_q4: This system is clearer than existing eco-labels
- post_q5: Feedback on personal purchases was useful
- post_q6: Would use this information for future choices
- post_q7: Quizzes helped me learn about environmental impact

In [ ]:
# Questionnaire response analysis
pre_q_cols = [c for c in df.columns if c.startswith('pre_q') and c[5:].isdigit()]
post_q_cols = [c for c in df.columns if c.startswith('post_q') and c[6:].isdigit()]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pre-questionnaire
if pre_q_cols:
    pre_means = df[pre_q_cols].mean()
    pre_stds = df[pre_q_cols].std()
    
    x = range(len(pre_q_cols))
    axes[0].bar(x, pre_means, yerr=pre_stds, capsize=3, color='coral', alpha=0.7, edgecolor='black')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([c.replace('pre_', '') for c in pre_q_cols])
    axes[0].set_ylabel('Mean Score (1-5 Likert)')
    axes[0].set_title('Pre-Questionnaire Responses')
    axes[0].set_ylim(1, 5.5)
    axes[0].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Neutral (3)')

# Post-questionnaire
if post_q_cols:
    post_means = df[post_q_cols].mean()
    post_stds = df[post_q_cols].std()
    
    x = range(len(post_q_cols))
    axes[1].bar(x, post_means, yerr=post_stds, capsize=3, color='teal', alpha=0.7, edgecolor='black')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([c.replace('post_', '') for c in post_q_cols])
    axes[1].set_ylabel('Mean Score (1-5 Likert)')
    axes[1].set_title('Post-Questionnaire Responses')
    axes[1].set_ylim(1, 5.5)
    axes[1].axhline(y=3, color='gray', linestyle='--', alpha=0.5, label='Neutral (3)')

plt.tight_layout()
plt.show()

# Summary statistics
print("=== Questionnaire Summary Statistics ===")
print("\nPre-Questionnaire:")
print(df[pre_q_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])
print("\nPost-Questionnaire:")
print(df[post_q_cols].describe().round(2).T[['mean', 'std', 'min', 'max']])

## 9. Qualitative Responses

Open-ended reflection responses for thematic analysis.

In [ ]:
# Extract qualitative responses from raw data
qualitative_responses = []

for session in sessions:
    reflection = session.get('_post_questionnaire_open') or session.get('_reflection') or {}
    if reflection and isinstance(reflection, dict):
        entry = {
            'session_id': session.get('id', '')[:8],  # First 8 chars for anonymity
            'diet': session.get('demo_diet'),
            'generic_improvement': session.get('generic_improvement'),
        }
        # Extract text responses
        for key, value in reflection.items():
            if isinstance(value, str) and value.strip():
                entry[key] = value.strip()
        
        if len(entry) > 3:  # Has actual text responses
            qualitative_responses.append(entry)

print(f"Found {len(qualitative_responses)} responses with open-ended feedback\n")

# Display responses by question type
question_keys = ['ref_reflection', 'ref_surprise', 'ref_system_feedback', 
                 'ref_trust_comparison', 'ref_improvement']

for key in question_keys:
    responses_for_key = [(r['session_id'], r.get(key)) for r in qualitative_responses if r.get(key)]
    if responses_for_key:
        print(f"{'=' * 70}")
        print(f"📝 {key.replace('ref_', '').replace('_', ' ').upper()}")
        print(f"   ({len(responses_for_key)} responses)")
        print(f"{'=' * 70}")
        for session_id, response in responses_for_key[:5]:  # Show first 5
            print(f"\n[{session_id}]: {response[:300]}{'...' if len(response) > 300 else ''}")
        if len(responses_for_key) > 5:
            print(f"\n... and {len(responses_for_key) - 5} more responses")
        print()

## 10. Export Data

Save processed data for further analysis (SPSS, Excel, etc.)

In [ ]:
# Export options
OUTPUT_DIR = "analysis_output"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save flat DataFrame to CSV (for SPSS/Excel)
csv_path = f"{OUTPUT_DIR}/experiment_data.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Saved: {csv_path}")

# 2. Save qualitative responses to JSON
qual_path = f"{OUTPUT_DIR}/qualitative_responses.json"
with open(qual_path, 'w', encoding='utf-8') as f:
    json.dump(qualitative_responses, f, indent=2, ensure_ascii=False)
print(f"✅ Saved: {qual_path}")

# 3. Save statistical results
stats_df = pd.DataFrame(results)
stats_path = f"{OUTPUT_DIR}/statistical_results.csv"
stats_df.to_csv(stats_path, index=False)
print(f"✅ Saved: {stats_path}")

# 4. Save full raw data (with nested quiz details)
full_path = f"{OUTPUT_DIR}/experiment_data_full.json"
with open(full_path, 'w', encoding='utf-8') as f:
    json.dump(raw_data, f, indent=2, default=str)
print(f"✅ Saved: {full_path}")

print(f"\n📁 All files saved to: {os.path.abspath(OUTPUT_DIR)}")

## Summary

Key findings from this analysis:

1. **Learning Effect**: Check the paired t-test results above to see if scores improved significantly
2. **Effect Size**: Cohen's d indicates the practical significance of the improvement
3. **Demographics**: Review if certain groups benefit more from the intervention
4. **User Feedback**: Qualitative responses provide insight into user experience

---
*Generated by experiment_analysis.ipynb*

## Reversed-Order Detection\n\nSome participants misread the task and sorted **LOWEST → HIGHEST** instead of **HIGHEST → LOWEST**.\nThis cell detects sessions where a user's ranking is strongly inversely correlated with the correct order (i.e. they reversed it).\n\nFor each quiz the stored `correctOrder` array gives the ground truth positions. We compute the Spearman rank correlation between the user's submitted order and the correct order:\n- **≈ +1.0** → user got it right\n- **≈ −1.0** → user fully reversed the order\n- near 0 → random / unsure\n\nSessions with correlation < −0.7 on any quiz are flagged.

In [3]:
import math
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv
from supabase import create_client

load_dotenv(Path.cwd().parent / '.env')
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_SERVICE_ROLE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError('Missing SUPABASE_URL or SUPABASE_SERVICE_ROLE_KEY in environment')

sb = create_client(SUPABASE_URL, SUPABASE_KEY)

QUIZ_KEYS  = ['quiz1_data', 'quiz2_data', 'quiz3_data', 'quiz4_data', 'quiz5_data', 'quiz6_data']
QUIZ_LABEL = {
    'quiz1_data': 'Q1 – General (pre)',
    'quiz2_data': 'Q3 – Personal (pre)',
    'quiz3_data': 'Q4 – General (post)',
    'quiz4_data': 'Q6 – Personal (post)',
    'quiz5_data': 'Q2 – AH (pre)',
    'quiz6_data': 'Q5 – AH (post)',
}
REVERSAL_THRESHOLD = -0.7

def rankdata(values):
    sorted_pairs = sorted(enumerate(values), key=lambda x: x[1])
    ranks = [0.0] * len(values)
    i = 0
    while i < len(sorted_pairs):
        j = i
        while j + 1 < len(sorted_pairs) and sorted_pairs[j + 1][1] == sorted_pairs[i][1]:
            j += 1
        avg_rank = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[sorted_pairs[k][0]] = avg_rank
        i = j + 1
    return ranks

def pearsonr(x, y):
    n = len(x)
    if n < 2:
        return None
    mean_x = sum(x) / n
    mean_y = sum(y) / n
    num = sum((a - mean_x) * (b - mean_y) for a, b in zip(x, y))
    den_x = math.sqrt(sum((a - mean_x) ** 2 for a in x))
    den_y = math.sqrt(sum((b - mean_y) ** 2 for b in y))
    if den_x == 0 or den_y == 0:
        return None
    return num / (den_x * den_y)

def spearmanr_simple(x, y):
    if len(x) != len(y) or len(x) < 2:
        return None
    return pearsonr(rankdata(x), rankdata(y))

def spearman_vs_correct(quiz_data):
    if not quiz_data:
        return None
    user_ranking = quiz_data.get('user_ranking') or []
    correct_order = quiz_data.get('correctOrder') or []
    if len(user_ranking) < 3 or len(correct_order) < 3:
        return None
    correct_pos = {item['id']: i for i, item in enumerate(correct_order)}
    user_pos = {item['id']: i for i, item in enumerate(user_ranking)}
    common_ids = [k for k in correct_pos if k in user_pos]
    if len(common_ids) < 3:
        return None
    c = [correct_pos[k] for k in common_ids]
    u = [user_pos[k] for k in common_ids]
    r = spearmanr_simple(c, u)
    return None if r is None else round(r, 3)

resp = sb.table('experiment_sessions').select('id, bonus_card, current_step, demographics, ' + ', '.join(QUIZ_KEYS)).execute()
sessions = resp.data
print(f'Fetched {len(sessions)} sessions')

reversed_sessions = []
for s in sessions:
    result = {
        'session_id': s['id'],
        'bonus_card': s.get('bonus_card', '—'),
        'current_step': s.get('current_step'),
        'quiz_correlations': {},
        'reversed_in': []
    }
    for key in QUIZ_KEYS:
        r = spearman_vs_correct(s.get(key))
        result['quiz_correlations'][QUIZ_LABEL[key]] = r
        if r is not None and r < REVERSAL_THRESHOLD:
            result['reversed_in'].append(QUIZ_LABEL[key])
    if result['reversed_in']:
        reversed_sessions.append(result)

print(f'\n{len(reversed_sessions)} sessions with suspected reversed rankings:\n')
pprint(reversed_sessions[:20])

# Keep this for the next cell
reversed_sessions

Fetched 54 sessions

20 sessions with suspected reversed rankings:

[{'bonus_card': 'CART_986a0521e05b',
  'current_step': 'complete',
  'quiz_correlations': {'Q1 – General (pre)': -0.943,
                        'Q2 – AH (pre)': -0.657,
                        'Q3 – Personal (pre)': -0.829,
                        'Q4 – General (post)': 0.943,
                        'Q5 – AH (post)': 0.314,
                        'Q6 – Personal (post)': 0.943},
  'reversed_in': ['Q1 – General (pre)', 'Q3 – Personal (pre)'],
  'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4'},
 {'bonus_card': 'CART_5458bf54714b',
  'current_step': 'complete',
  'quiz_correlations': {'Q1 – General (pre)': -0.771,
                        'Q2 – AH (pre)': 0.486,
                        'Q3 – Personal (pre)': 0.2,
                        'Q4 – General (post)': -0.029,
                        'Q5 – AH (post)': 0.714,
                        'Q6 – Personal (post)': 0.943},
  'reversed_in': ['Q1 – General (pre)'],
  'se

[{'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4',
  'bonus_card': 'CART_986a0521e05b',
  'current_step': 'complete',
  'quiz_correlations': {'Q1 – General (pre)': -0.943,
   'Q3 – Personal (pre)': -0.829,
   'Q4 – General (post)': 0.943,
   'Q6 – Personal (post)': 0.943,
   'Q2 – AH (pre)': -0.657,
   'Q5 – AH (post)': 0.314},
  'reversed_in': ['Q1 – General (pre)', 'Q3 – Personal (pre)']},
 {'session_id': '5458bf54-714b-4102-a0ad-6d8cc48a7290',
  'bonus_card': 'CART_5458bf54714b',
  'current_step': 'complete',
  'quiz_correlations': {'Q1 – General (pre)': -0.771,
   'Q3 – Personal (pre)': 0.2,
   'Q4 – General (post)': -0.029,
   'Q6 – Personal (post)': 0.943,
   'Q2 – AH (pre)': 0.486,
   'Q5 – AH (post)': 0.714},
  'reversed_in': ['Q1 – General (pre)']},
 {'session_id': '9d8f705f-bffe-42a3-8668-0309b584f7f6',
  'bonus_card': 'CART_9d8f705fbffe',
  'current_step': 'complete',
  'quiz_correlations': {'Q1 – General (pre)': -1.0,
   'Q3 – Personal (pre)': -0.771,
   'Q4 – General 

In [ ]:
## ── Fix reversed rankings ────────────────────────────────────────────────────
# Set DRY_RUN = True to preview what would change without writing to the DB.
# Set DRY_RUN = False to actually apply the corrections.
DRY_RUN = True

# Optionally restrict to specific session IDs (leave empty to fix ALL flagged sessions)
FIX_ONLY_SESSIONS = []  # e.g. ['abc-123', 'def-456']

def distance_score(user_order, correct_order):
    """Recompute score: sum of |position differences|, normalised to 0-100."""
    correct_pos = {item['id']: i for i, item in enumerate(correct_order)}
    n = len(correct_order)
    total_dist = sum(abs(correct_pos.get(item['id'], i) - i) for i, item in enumerate(user_order))
    # Max possible distance for n items reversed
    max_dist = sum(abs(i - (n - 1 - i)) for i in range(n))
    if max_dist == 0:
        return 100
    return round((1 - total_dist / max_dist) * 100)

to_fix = df_reversed['session_id'].tolist()
if FIX_ONLY_SESSIONS:
    to_fix = [sid for sid in to_fix if sid in FIX_ONLY_SESSIONS]

print(f"{'[DRY RUN] ' if DRY_RUN else ''}Fixing {len(to_fix)} session(s)...\n")

for sid in to_fix:
    session = next((s for s in sessions if s['id'] == sid), None)
    if not session: continue
    updates = {}
    for key in QUIZ_KEYS:
        qd = session.get(key)
        if not qd: continue
        r = spearman_vs_correct(qd)
        if r is None or r >= REVERSAL_THRESHOLD:
            continue  # Not reversed — skip

        user_ranking  = qd.get('user_ranking', [])
        correct_order = qd.get('correctOrder', [])
        items         = qd.get('items', [])

        # Reverse the submitted ranking
        fixed_ranking = list(reversed(user_ranking))
        new_score = distance_score(fixed_ranking, correct_order)

        updates[key] = {
            **qd,
            'user_ranking':   fixed_ranking,
            'score':          new_score,
            'reversed_correction': True,
        }
        print(f"  {sid[:8]}… {QUIZ_LABEL[key]}: "
              f"score {qd.get('score')} → {new_score}  (r={r})")

    if updates and not DRY_RUN:
        sb.table('experiment_sessions').update(updates).eq('id', sid).execute()
        print(f"  ✓ Written to DB")
    elif updates and DRY_RUN:
        print(f"  [DRY RUN] Would update {len(updates)} quiz(zes) for session {sid[:8]}…")

print("\nDone. Set DRY_RUN = False to apply changes.")

## Focused accidental-misread detection\n\nThis section narrows the broader reversed-ranking detection into likely **instruction misreads**:\n\n- **Q1 misreaders**: reversed on the very first sorting task, but not generally reversed later\n- **First-three misreaders**: reversed on at least 2 of the first 3 sorting tasks, but mostly corrected themselves in the post-intervention quizzes\n\nExperiment order is: **Q1 → Q2 AH (= quiz5) → Q3 personal (= quiz2) → Q4 → Q5 → Q6**.

In [4]:
PRE_ORDER = ['quiz1_data', 'quiz5_data', 'quiz2_data']   # actual first three sorting games
POST_ORDER = ['quiz3_data', 'quiz6_data', 'quiz4_data']  # actual last three sorting games

POSITIVE_THRESHOLD = 0.3
NEGATIVE_THRESHOLD = REVERSAL_THRESHOLD

def quiz_corr_map(session):
    return {key: spearman_vs_correct(session.get(key)) for key in QUIZ_KEYS}

def is_reversed(r):
    return r is not None and r < NEGATIVE_THRESHOLD

def is_positive(r):
    return r is not None and r > POSITIVE_THRESHOLD

def summarize_session(session):
    corrs = quiz_corr_map(session)
    pre_reversed = [QUIZ_LABEL[k] for k in PRE_ORDER if is_reversed(corrs[k])]
    post_positive = [QUIZ_LABEL[k] for k in POST_ORDER if is_positive(corrs[k])]
    post_reversed = [QUIZ_LABEL[k] for k in POST_ORDER if is_reversed(corrs[k])]
    return {
        'session_id': session['id'],
        'bonus_card': session.get('bonus_card', '—'),
        'current_step': session.get('current_step'),
        'pre_reversed': pre_reversed,
        'post_positive': post_positive,
        'post_reversed': post_reversed,
        'corrs': {QUIZ_LABEL[k]: corrs[k] for k in QUIZ_KEYS},
    }

# Bucket 1: likely Q1-only / Q1-first misread
# Heuristic: Q1 reversed, but at least 2 post quizzes are positive and at most 1 post quiz reversed.
q1_misreaders = []
for s in sessions:
    summary = summarize_session(s)
    q1_r = summary['corrs']['Q1 – General (pre)']
    if is_reversed(q1_r) and len(summary['post_positive']) >= 2 and len(summary['post_reversed']) <= 1:
        q1_misreaders.append(summary)

# Bucket 2: likely first-three misreaders
# Heuristic: at least 2 of first 3 reversed, but at least 2 of last 3 positive.
first_three_misreaders = []
for s in sessions:
    summary = summarize_session(s)
    if len(summary['pre_reversed']) >= 2 and len(summary['post_positive']) >= 2:
        first_three_misreaders.append(summary)

print('Likely Q1 misreaders (reversed Q1, later behavior mostly corrected):')
for row in q1_misreaders:
    print(f"- {row['session_id']} | bonus_card={row['bonus_card']} | post_positive={row['post_positive']}")
print(f"Total: {len(q1_misreaders)}\n")

print('Likely first-three misreaders (>=2 reversed in first 3, >=2 positive in last 3):')
for row in first_three_misreaders:
    print(f"- {row['session_id']} | bonus_card={row['bonus_card']} | pre_reversed={row['pre_reversed']} | post_positive={row['post_positive']}")
print(f"Total: {len(first_three_misreaders)}\n")

focused_misread_output = {
    'q1_misreaders': q1_misreaders,
    'first_three_misreaders': first_three_misreaders,
}
focused_misread_output

Likely Q1 misreaders (reversed Q1, later behavior mostly corrected):
- 986a0521-e05b-4afd-ab17-77ab9a5a87b4 | bonus_card=CART_986a0521e05b | post_positive=['Q4 – General (post)', 'Q5 – AH (post)', 'Q6 – Personal (post)']
- 5458bf54-714b-4102-a0ad-6d8cc48a7290 | bonus_card=CART_5458bf54714b | post_positive=['Q5 – AH (post)', 'Q6 – Personal (post)']
- bb3a59a5-5a7d-4813-8333-3b098a8e437a | bonus_card=CART_bb3a59a55a7d | post_positive=['Q4 – General (post)', 'Q5 – AH (post)']
- 2484e621-9f6f-47c6-bf9c-9e8df42c3601 | bonus_card=CART_2484e6219f6f | post_positive=['Q4 – General (post)', 'Q5 – AH (post)', 'Q6 – Personal (post)']
- 68661d30-550b-4e2f-bda2-ab3de5d155f0 | bonus_card=2622185336743 | post_positive=['Q4 – General (post)', 'Q5 – AH (post)', 'Q6 – Personal (post)']
- dca22322-ed10-4ea1-b384-16280e1988a9 | bonus_card=CART_dca22322ed10 | post_positive=['Q4 – General (post)', 'Q5 – AH (post)', 'Q6 – Personal (post)']
- d309068f-051c-498a-b081-2b0a31c3bc07 | bonus_card=2620678981951 | po

{'q1_misreaders': [{'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4',
   'bonus_card': 'CART_986a0521e05b',
   'current_step': 'complete',
   'pre_reversed': ['Q1 – General (pre)', 'Q3 – Personal (pre)'],
   'post_positive': ['Q4 – General (post)',
    'Q5 – AH (post)',
    'Q6 – Personal (post)'],
   'post_reversed': [],
   'corrs': {'Q1 – General (pre)': -0.943,
    'Q3 – Personal (pre)': -0.829,
    'Q4 – General (post)': 0.943,
    'Q6 – Personal (post)': 0.943,
    'Q2 – AH (pre)': -0.657,
    'Q5 – AH (post)': 0.314}},
  {'session_id': '5458bf54-714b-4102-a0ad-6d8cc48a7290',
   'bonus_card': 'CART_5458bf54714b',
   'current_step': 'complete',
   'pre_reversed': ['Q1 – General (pre)'],
   'post_positive': ['Q5 – AH (post)', 'Q6 – Personal (post)'],
   'post_reversed': [],
   'corrs': {'Q1 – General (pre)': -0.771,
    'Q3 – Personal (pre)': 0.2,
    'Q4 – General (post)': -0.029,
    'Q6 – Personal (post)': 0.943,
    'Q2 – AH (pre)': 0.486,
    'Q5 – AH (post)': 0.714}},
  {'

In [5]:
# Which likely Q1 misreaders put beef first?

BEEF_NAMES = {'ground beef', 'rundergehakt'}

q1_beef_first = []
for row in q1_misreaders:
    session = next((s for s in sessions if s['id'] == row['session_id']), None)
    if not session:
        continue
    quiz1 = session.get('quiz1_data') or {}
    user_ranking = quiz1.get('user_ranking') or []
    if not user_ranking:
        continue
    first_item = user_ranking[0]
    first_name = (first_item.get('name') or '').strip().lower()
    if first_name in BEEF_NAMES:
        q1_beef_first.append({
            'session_id': row['session_id'],
            'bonus_card': row['bonus_card'],
            'first_item': first_item.get('name'),
            'q1_corr': row['corrs']['Q1 – General (pre)'],
            'pre_reversed': row['pre_reversed'],
            'post_positive': row['post_positive'],
        })

print('Likely Q1 misreaders who put beef first:')
for item in q1_beef_first:
    print(f"- {item['session_id']} | bonus_card={item['bonus_card']} | first_item={item['first_item']} | q1_corr={item['q1_corr']}")
print(f"Total: {len(q1_beef_first)}")

q1_beef_first

Likely Q1 misreaders who put beef first:
- 986a0521-e05b-4afd-ab17-77ab9a5a87b4 | bonus_card=CART_986a0521e05b | first_item=Ground Beef | q1_corr=-0.943
- bb3a59a5-5a7d-4813-8333-3b098a8e437a | bonus_card=CART_bb3a59a55a7d | first_item=Ground Beef | q1_corr=-0.771
- 2484e621-9f6f-47c6-bf9c-9e8df42c3601 | bonus_card=CART_2484e6219f6f | first_item=Ground Beef | q1_corr=-0.829
- 68661d30-550b-4e2f-bda2-ab3de5d155f0 | bonus_card=2622185336743 | first_item=Ground Beef | q1_corr=-0.771
- d309068f-051c-498a-b081-2b0a31c3bc07 | bonus_card=2620678981951 | first_item=Ground Beef | q1_corr=-0.886
- 4992851d-3a88-46a0-8e26-3e186483f2b1 | bonus_card=CART_4992851d3a88 | first_item=Ground Beef | q1_corr=-0.829
- 9ddb493d-77a4-4840-8ac4-bbeeefdc3901 | bonus_card=CART_9ddb493d77a4 | first_item=Ground Beef | q1_corr=-0.886
- 7efce358-e163-4958-ad77-3d6164074d51 | bonus_card=CART_7efce358e163 | first_item=Ground Beef | q1_corr=-0.829
- 0f988ea0-aca3-46fd-8754-9abd502f3886 | bonus_card=CART_0f988ea0aca3 |

[{'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4',
  'bonus_card': 'CART_986a0521e05b',
  'first_item': 'Ground Beef',
  'q1_corr': -0.943,
  'pre_reversed': ['Q1 – General (pre)', 'Q3 – Personal (pre)'],
  'post_positive': ['Q4 – General (post)',
   'Q5 – AH (post)',
   'Q6 – Personal (post)']},
 {'session_id': 'bb3a59a5-5a7d-4813-8333-3b098a8e437a',
  'bonus_card': 'CART_bb3a59a55a7d',
  'first_item': 'Ground Beef',
  'q1_corr': -0.771,
  'pre_reversed': ['Q1 – General (pre)'],
  'post_positive': ['Q4 – General (post)', 'Q5 – AH (post)']},
 {'session_id': '2484e621-9f6f-47c6-bf9c-9e8df42c3601',
  'bonus_card': 'CART_2484e6219f6f',
  'first_item': 'Ground Beef',
  'q1_corr': -0.829,
  'pre_reversed': ['Q1 – General (pre)'],
  'post_positive': ['Q4 – General (post)',
   'Q5 – AH (post)',
   'Q6 – Personal (post)']},
 {'session_id': '68661d30-550b-4e2f-bda2-ab3de5d155f0',
  'bonus_card': '2622185336743',
  'first_item': 'Ground Beef',
  'q1_corr': -0.771,
  'pre_reversed': ['Q1 – G

In [6]:
# Users who put a meat product first in pre-quiz general (Q1) and pre-quiz AH (Q5)

MEATISH_NAMES = {
    'ground beef', 'rundergehakt',
    'beef', 'rundvlees',
    'chicken breast', 'kipfilet',
    'lamb', 'lamsvlees',
    'pork', 'varkensvlees',
    'steak', 'biefstuk'
}

def first_item_name(session, quiz_key):
    quiz = session.get(quiz_key) or {}
    ranking = quiz.get('user_ranking') or []
    if not ranking:
        return None
    return (ranking[0].get('name') or '').strip()

def is_meat_name(name):
    return bool(name) and name.strip().lower() in MEATISH_NAMES

q1_meat_first = []
q5_meat_first = []

for session in sessions:
    q1_name = first_item_name(session, 'quiz1_data')
    q5_name = first_item_name(session, 'quiz5_data')

    if is_meat_name(q1_name):
        q1_meat_first.append({
            'session_id': session['id'],
            'bonus_card': session.get('bonus_card', '—'),
            'first_item': q1_name,
            'q1_corr': spearman_vs_correct(session.get('quiz1_data')),
        })

    if is_meat_name(q5_name):
        q5_meat_first.append({
            'session_id': session['id'],
            'bonus_card': session.get('bonus_card', '—'),
            'first_item': q5_name,
            'q5_corr': spearman_vs_correct(session.get('quiz5_data')),
        })

print('Q1 (pre-quiz general): users who put meat first')
for row in q1_meat_first:
    print(f"- {row['session_id']} | bonus_card={row['bonus_card']} | first_item={row['first_item']} | q1_corr={row['q1_corr']}")
print(f"Total: {len(q1_meat_first)}\n")

print('Q5 (pre-quiz AH): users who put meat first')
for row in q5_meat_first:
    print(f"- {row['session_id']} | bonus_card={row['bonus_card']} | first_item={row['first_item']} | q5_corr={row['q5_corr']}")
print(f"Total: {len(q5_meat_first)}\n")

if len(q5_meat_first) == 0:
    print('Note: quiz 5 contains no meat products in AH_POOL_C, so this is expected.')

{
    'q1_meat_first': q1_meat_first,
    'q5_meat_first': q5_meat_first,
}

Q1 (pre-quiz general): users who put meat first
- 1f25fd9a-a759-4d2f-a687-08b8ac1f19d5 | bonus_card=2621941614590 | first_item=Ground Beef | q1_corr=0.029
- 986a0521-e05b-4afd-ab17-77ab9a5a87b4 | bonus_card=CART_986a0521e05b | first_item=Ground Beef | q1_corr=-0.943
- 3b4aacff-b057-4e0f-b499-d7ca5f178ed4 | bonus_card=2621621505101 | first_item=Chicken Breast | q1_corr=0.486
- 03448d58-d95b-44ef-88fc-561eabd272e3 | bonus_card=CART_03448d58d95b | first_item=Ground Beef | q1_corr=-0.657
- 9d8f705f-bffe-42a3-8668-0309b584f7f6 | bonus_card=CART_9d8f705fbffe | first_item=Ground Beef | q1_corr=-1.0
- bb3a59a5-5a7d-4813-8333-3b098a8e437a | bonus_card=CART_bb3a59a55a7d | first_item=Ground Beef | q1_corr=-0.771
- c26dbd3b-35c2-4e2e-b61f-407833d573d3 | bonus_card=CART_c26dbd3b35c2 | first_item=Chicken Breast | q1_corr=0.029
- 2484e621-9f6f-47c6-bf9c-9e8df42c3601 | bonus_card=CART_2484e6219f6f | first_item=Ground Beef | q1_corr=-0.829
- 258f6c03-ddb7-4106-94a8-11a8b587595c | bonus_card=CART_258f6c

{'q1_meat_first': [{'session_id': '1f25fd9a-a759-4d2f-a687-08b8ac1f19d5',
   'bonus_card': '2621941614590',
   'first_item': 'Ground Beef',
   'q1_corr': 0.029},
  {'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4',
   'bonus_card': 'CART_986a0521e05b',
   'first_item': 'Ground Beef',
   'q1_corr': -0.943},
  {'session_id': '3b4aacff-b057-4e0f-b499-d7ca5f178ed4',
   'bonus_card': '2621621505101',
   'first_item': 'Chicken Breast',
   'q1_corr': 0.486},
  {'session_id': '03448d58-d95b-44ef-88fc-561eabd272e3',
   'bonus_card': 'CART_03448d58d95b',
   'first_item': 'Ground Beef',
   'q1_corr': -0.657},
  {'session_id': '9d8f705f-bffe-42a3-8668-0309b584f7f6',
   'bonus_card': 'CART_9d8f705fbffe',
   'first_item': 'Ground Beef',
   'q1_corr': -1.0},
  {'session_id': 'bb3a59a5-5a7d-4813-8333-3b098a8e437a',
   'bonus_card': 'CART_bb3a59a55a7d',
   'first_item': 'Ground Beef',
   'q1_corr': -0.771},
  {'session_id': 'c26dbd3b-35c2-4e2e-b61f-407833d573d3',
   'bonus_card': 'CART_c26dbd3b35c2

In [7]:
# Full Q1 ranking dump for all sessions that put meat first
# Useful for manual diagnosis of likely accidental reversals.

q1_meat_first_rankings = []

for row in q1_meat_first:
    session = next((s for s in sessions if s['id'] == row['session_id']), None)
    if not session:
        continue

    quiz1 = session.get('quiz1_data') or {}
    user_ranking = quiz1.get('user_ranking') or []
    correct_order = quiz1.get('correctOrder') or []

    record = {
        'session_id': row['session_id'],
        'bonus_card': row['bonus_card'],
        'q1_corr': row['q1_corr'],
        'user_ranking': [item.get('name') for item in user_ranking],
        'correct_order': [item.get('name') for item in correct_order],
    }
    q1_meat_first_rankings.append(record)

for rec in q1_meat_first_rankings:
    print('=' * 100)
    print(f"SESSION: {rec['session_id']}")
    print(f"BONUS CARD: {rec['bonus_card']}")
    print(f"Q1 CORRELATION: {rec['q1_corr']}")
    print('USER RANKING   :', '  >  '.join(rec['user_ranking']))
    print('CORRECT RANKING:', '  >  '.join(rec['correct_order']))
    print()

q1_meat_first_rankings

SESSION: 1f25fd9a-a759-4d2f-a687-08b8ac1f19d5
BONUS CARD: 2621941614590
Q1 CORRELATION: 0.029
USER RANKING   : Ground Beef  >  Rice  >  Orange  >  Milk  >  Coffee  >  Chicken Breast
CORRECT RANKING: Orange  >  Rice  >  Milk  >  Chicken Breast  >  Coffee  >  Ground Beef

SESSION: 986a0521-e05b-4afd-ab17-77ab9a5a87b4
BONUS CARD: CART_986a0521e05b
Q1 CORRELATION: -0.943
USER RANKING   : Ground Beef  >  Chicken Breast  >  Coffee  >  Milk  >  Rice  >  Orange
CORRECT RANKING: Orange  >  Rice  >  Milk  >  Chicken Breast  >  Coffee  >  Ground Beef

SESSION: 3b4aacff-b057-4e0f-b499-d7ca5f178ed4
BONUS CARD: 2621621505101
Q1 CORRELATION: 0.486
USER RANKING   : Chicken Breast  >  Rice  >  Orange  >  Coffee  >  Milk  >  Ground Beef
CORRECT RANKING: Orange  >  Rice  >  Milk  >  Chicken Breast  >  Coffee  >  Ground Beef

SESSION: 03448d58-d95b-44ef-88fc-561eabd272e3
BONUS CARD: CART_03448d58d95b
Q1 CORRELATION: -0.657
USER RANKING   : Ground Beef  >  Chicken Breast  >  Milk  >  Rice  >  Coffee  >  Or

[{'session_id': '1f25fd9a-a759-4d2f-a687-08b8ac1f19d5',
  'bonus_card': '2621941614590',
  'q1_corr': 0.029,
  'user_ranking': ['Ground Beef',
   'Rice',
   'Orange',
   'Milk',
   'Coffee',
   'Chicken Breast'],
  'correct_order': ['Orange',
   'Rice',
   'Milk',
   'Chicken Breast',
   'Coffee',
   'Ground Beef']},
 {'session_id': '986a0521-e05b-4afd-ab17-77ab9a5a87b4',
  'bonus_card': 'CART_986a0521e05b',
  'q1_corr': -0.943,
  'user_ranking': ['Ground Beef',
   'Chicken Breast',
   'Coffee',
   'Milk',
   'Rice',
   'Orange'],
  'correct_order': ['Orange',
   'Rice',
   'Milk',
   'Chicken Breast',
   'Coffee',
   'Ground Beef']},
 {'session_id': '3b4aacff-b057-4e0f-b499-d7ca5f178ed4',
  'bonus_card': '2621621505101',
  'q1_corr': 0.486,
  'user_ranking': ['Chicken Breast',
   'Rice',
   'Orange',
   'Coffee',
   'Milk',
   'Ground Beef'],
  'correct_order': ['Orange',
   'Rice',
   'Milk',
   'Chicken Breast',
   'Coffee',
   'Ground Beef']},
 {'session_id': '03448d58-d95b-44ef-88